In [1]:
import os, glob, json, re
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from google.colab import drive
drive.mount("/content/drive")

hits = glob.glob("/content/drive/MyDrive/**/data/features.parquet", recursive=True)
print("found:", hits)
FEAT = hits[0]                                        # change the index if several copies are listed
ROOT = FEAT.rsplit("/data/", 1)[0]
F = pd.read_parquet(FEAT)
P = pd.DataFrame([json.loads(l) for l in open(f"{ROOT}/data/passages.jsonl")])
P = P[P["context_id"].notna()][["context_id", "rank", "passage", "question"]]
D = F.merge(P, on=["context_id", "rank"], how="left", validate="one_to_one")
assert D["passage"].notna().all() and len(D) == len(F)


Mounted at /content/drive
found: ['/content/drive/MyDrive/rag_trust/data/features.parquet']


In [2]:
# text baselines
SENT_END = re.compile(r"[.!?][\"'\)\]]*$")
D["n_words"] = D["passage"].str.split().str.len()
D["starts_q"] = [float(p.startswith(q + ".")) for p, q in zip(D["passage"], D["question"])]
D["full_stop"] = [float(bool(p.split()) and bool(SENT_END.search(p.split()[-1]))) for p in D["passage"]]

# ---- 1. baseline directions: set once on NON-augmented train rows (original n=0 and n=1 groups), then frozen
tr0 = D[(D.split == "train") & (D.group_variant == "original") & (D.n_poison.isin([0, 1]))]
DIRECTION = {}
for name, col, sign in [("length", "n_words", -1), ("retrieval_score", "f1", 1)]:
    auc = roc_auc_score(tr0["label"], sign * tr0[col])
    DIRECTION[name] = sign if auc >= 0.5 else -sign
    print(f"{name}: train AUROC {auc:.3f} in the expected direction -> frozen direction {DIRECTION[name]:+d}")
D["s_length"] = DIRECTION["length"] * D["n_words"]
D["s_retrieval"] = DIRECTION["retrieval_score"] * D["f1"]

# ---- 2. feature scorer: train_aug groups of train questions, n=0 and the five n=1 groups (n=5 excluded)
tr = D[(D.split == "train") & (D.group_variant == "train_aug") & (D.n_poison.isin([0, 1]))]
print("feature scorer training rows:", len(tr), "| poisoned:", int(tr["label"].sum()))
scaler = StandardScaler().fit(tr[["f1", "f2", "f3"]])
lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000).fit(scaler.transform(tr[["f1", "f2", "f3"]]), tr["label"])
print("coefficients (standardised f1, f2, f3):", lr.coef_.round(3), "| intercept:", lr.intercept_.round(3))
D["s_featscorer"] = lr.decision_function(scaler.transform(D[["f1", "f2", "f3"]]))

# ---- 3. thresholds: 5% and 1% FPR on the val n=0 clean passages (NQ + HotpotQA), then frozen
val_clean = D[(D.split == "val") & (D.group_variant == "original") & (D.n_poison == 0) & (D.label == 0)]
val_pos = D[(D.split == "val") & (D.group_variant == "original") & (D.n_poison == 1) & (D.label == 1)]
print("val clean passages:", len(val_clean), "| val poisoned (n=1 groups):", len(val_pos))
THRESH = {}
for name, col in [("featscorer", "s_featscorer"), ("length", "s_length"), ("retrieval", "s_retrieval")]:
    THRESH[name] = {}
    for tag, fpr in [("fpr5", 0.05), ("fpr1", 0.01)]:
        thr = float(np.quantile(val_clean[col], 1 - fpr))            # flag when score > thr
        THRESH[name][tag] = {"threshold": thr,
                             "val_fpr_achieved": float((val_clean[col] > thr).mean()),
                             "val_tpr": float((val_pos[col] > thr).mean())}
        print(f"{name:11} {tag}: thr {thr:8.3f} | val FPR {THRESH[name][tag]['val_fpr_achieved']:.3f} | val TPR {THRESH[name][tag]['val_tpr']:.3f}")


length: train AUROC 0.723 in the expected direction -> frozen direction -1
retrieval_score: train AUROC 0.985 in the expected direction -> frozen direction +1
feature scorer training rows: 3595 | poisoned: 599
coefficients (standardised f1, f2, f3): [[0.855 0.215 0.247]] | intercept: [-0.283]
val clean passages: 200 | val poisoned (n=1 groups): 198
featscorer  fpr5: thr    0.122 | val FPR 0.050 | val TPR 0.980
featscorer  fpr1: thr    0.576 | val FPR 0.010 | val TPR 0.869
length      fpr5: thr  -10.000 | val FPR 0.045 | val TPR 0.000
length      fpr1: thr   -3.000 | val FPR 0.005 | val TPR 0.000
retrieval   fpr5: thr    1.267 | val FPR 0.050 | val TPR 0.970
retrieval   fpr1: thr    1.442 | val FPR 0.010 | val TPR 0.712


In [3]:
os.makedirs(f"{ROOT}/data/frozen", exist_ok=True)
joblib.dump({"scaler": scaler, "lr": lr}, f"{ROOT}/data/frozen/feature_scorer.joblib")
json.dump({"directions": DIRECTION, "thresholds": THRESH,
           "note": "set on val n=0 clean passages (NQ+HotpotQA); frozen for every test set, MS MARCO and every variant"},
          open(f"{ROOT}/data/frozen/thresholds.json", "w"), indent=2)
D[["dataset", "qid", "split", "context_id", "n_poison", "group_variant", "variant", "source", "rank", "label",
   "n_words", "starts_q", "full_stop", "s_length", "s_retrieval", "s_featscorer"]].to_parquet(f"{ROOT}/data/scores.parquet", index=False)
print("saved: frozen/feature_scorer.joblib, frozen/thresholds.json, scores.parquet")

saved: frozen/feature_scorer.joblib, frozen/thresholds.json, scores.parquet
